In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import asyncio
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import Audio
import pyquist as pq
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, STEEL

Drag the sliders to choose a sample: its offset $m / f_s$ and its duration
$M / f_s$, both in seconds. The sample is highlighted in blue, and the zoom
slider magnifies the waveform around it. The audio card underneath always
plays the current sample.

In [ ]:
# hide
# autorun
amen = pq.Audio.from_file("../assets/amen.wav").as_mono()
sr = amen.sample_rate
x = np.asarray(amen.samples).reshape(-1)
T = len(x) / sr
YMAX = 1.1 * np.abs(x).max()
OFFSET0, DURATION0, ZOOM0 = 1.744, 0.436, 1.0   # the first beat of the second bar


def view(lo, hi, x=x, sr=sr, max_pts=2000):
    # times and values of x[lo:hi], min/max-decimated so the peaks survive
    lo, hi = max(0, int(lo)), min(len(x), int(hi))
    seg = x[lo:hi]
    step = max(1, len(seg) // (max_pts // 2))
    n = len(seg) // step * step
    if step == 1 or n == 0:
        return (lo + np.arange(len(seg))) / sr, seg
    blocks = seg[:n].reshape(-1, step)
    t = (lo + np.arange(0, n, step) + step / 2) / sr
    return np.repeat(t, 2), np.column_stack([blocks.min(1), blocks.max(1)]).reshape(-1)


def window(offset, duration, zoom, T=T):
    # the visible time range: T / zoom wide, centered on the sample (when
    # zoomed in far enough, the sample can extend past both edges)
    width = T / zoom
    lo = min(max(0.0, offset + duration / 2 - width / 2), T - width)
    return lo, lo + width


def figure():
    lo, hi = window(OFFSET0, DURATION0, ZOOM0)
    a, b = OFFSET0, OFFSET0 + DURATION0
    t_all, y_all = view(lo * sr, hi * sr)
    t_seg, y_seg = view(a * sr, b * sr)
    fig = go.Figure()
    fig.add_scatter(x=[a, a, b, b, a], y=[-YMAX, YMAX, YMAX, -YMAX, -YMAX],
                    fill="toself", fillcolor="rgba(0, 123, 192, 0.12)",
                    line=dict(width=0), hoverinfo="skip")
    fig.add_scatter(x=t_all, y=y_all, mode="lines", line=dict(color=IRON, width=1))
    fig.add_scatter(x=t_seg, y=y_seg, mode="lines", line=dict(color=BLUE, width=1))
    fig.update_xaxes(range=[lo, hi], title_text="Time (s)", fixedrange=True)
    fig.update_yaxes(range=[-YMAX, YMAX], title_text="Amplitude", fixedrange=True)
    fig.update_layout(showlegend=False)
    return fig


def controls(fig):
    offset = widgets.FloatSlider(description="Offset (s)", min=0.0, max=round(T - 0.01, 3),
                                 step=0.001, value=OFFSET0, readout_format=".3f")
    duration = widgets.FloatSlider(description="Duration (s)", min=0.01, max=round(T, 3),
                                   step=0.001, value=DURATION0, readout_format=".3f")
    zoom = widgets.FloatSlider(description="Zoom", min=1.0, max=40.0, step=0.5,
                               value=ZOOM0, readout_format=".1f")
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(off, dur, z, view=view, window=window, sr=sr, T=T, N=len(x)):
        dur = min(dur, T - off)                  # a sample can't run past the end
        lo, hi = window(off, dur, z)
        t_all, y_all = view(lo * sr, hi * sr)
        t_seg, y_seg = view(off * sr, (off + dur) * sr)
        with fig.batch_update():
            fig.data[0].x = [off, off, off + dur, off + dur, off]
            fig.data[1].x, fig.data[1].y = t_all, y_all
            fig.data[2].x, fig.data[2].y = t_seg, y_seg
            fig.layout.xaxis.range = [lo, hi]
        readout.value = (f"<span style='font-size:0.9em'><i>m</i> = {int(off * sr):,} "
                         f"&nbsp;·&nbsp; <i>M</i> = {int(dur * sr):,} &nbsp;·&nbsp; "
                         f"<i>N</i> = {N:,} samples</span>")

    widgets.interactive_output(update, {"off": offset, "dur": duration, "z": zoom})

    # the audio card under the controls: the previous clip stays in place
    # while you drag (so the layout never jumps) and is swapped for the new
    # one when the pointer releases (keyboard nudges settle on a timer). It is
    # written through the Output's synced `outputs` trait, which works
    # outside a kernel message, where display() output has no destination
    out = widgets.Output()
    gate = icm_plotly.release_gate()   # pointer state: is a slider mid-drag?
    pending = []
    dirty = []

    def render(x=x, sr=sr, T=T):
        off = offset.value
        dur = min(duration.value, T - off)
        y = x[int(off * sr):int(off * sr) + int(dur * sr)].copy()
        n = min(len(y) // 2, int(0.002 * sr))      # 2 ms fades so the cuts don't click
        y[:n] *= np.linspace(0, 1, n)
        y[len(y) - n:] *= np.linspace(1, 0, n)
        y *= 0.125 / max(np.abs(y).max(), 1e-6)    # about -18 dBFS, a safe level
        audio = Audio(y.astype(np.float32), rate=sr, normalize=False)
        data, metadata = get_ipython().display_formatter.format(audio)
        # one assignment swaps the old card for the new one in place, so
        # the page never shows an empty card and nothing shifts
        out.outputs = ({"output_type": "display_data",
                        "data": data, "metadata": metadata},)

    async def settle():
        await asyncio.sleep(0.25)
        pending.clear()
        if dirty and not gate.dragging:
            dirty.clear()
            render()

    def on_change(_):
        dirty.append(True)
        if pending:
            pending.pop().cancel()
        pending.append(asyncio.ensure_future(settle()))

    def on_release(change):
        if not change["new"] and dirty:
            if pending:
                pending.pop().cancel()
            dirty.clear()
            render()

    gate.observe(on_release, names="dragging")

    for s in (offset, duration):                   # zooming doesn't change the sound
        s.observe(on_change, names="value")
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([offset, duration, zoom, readout, out, gate])

icm_plotly.show(figure, controls)